# Word2Vec

**Word2Vec** representa un salto monumental en la historia del Procesamiento de Lenguaje Natural (NLP). Si TF-IDF y PMI tratan las palabras como símbolos aislados y cuentan sus frecuencias, Word2Vec trata de **entender su significado** observando con quién se juntan.

Se basa en una premisa famosa del lingüista J.R. Firth (1957): *"Conocerás una palabra por la compañía que frecuenta"*.

Aquí tienes la explicación conceptual, cómo construye su aprendizaje (Toy Example) y cómo implementarlo en Python.

## 1. La Teoría: De conteos a vectores densos (Embeddings)

En modelos tradicionales (como TF-IDF), cada palabra es una dimensión diferente, lo que crea vectores inmensos y llenos de ceros (representaciones dispersas) donde "perro" y "gato" no tienen ninguna relación matemática, a pesar de ser mascotas.

**Word2Vec** resuelve esto usando una red neuronal de una sola capa oculta para comprimir el significado de las palabras en **vectores densos** (usualmente de 100 a 300 dimensiones). En este espacio geométrico, las palabras con significados similares terminan muy cerca unas de otras.

Esta geometría permite hacer "álgebra de palabras". El ejemplo más famoso de Word2Vec es:
$$\vec{v}_{\text{rey}} - \vec{v}_{\text{hombre}} + \vec{v}_{\text{mujer}} \approx \vec{v}_{\text{reina}}$$

### Las dos arquitecturas de Word2Vec

Para entrenar esta red neuronal, Word2Vec puede jugar a dos "juegos" diferentes:

1.  **CBOW (Continuous Bag of Words):** Intenta adivinar una palabra oculta basándose en su contexto. (Ej: "El ___ persigue al ratón" $\rightarrow$ adivinar "gato").
2.  **Skip-gram:** Hace lo contrario. Dada una palabra central, intenta adivinar cuáles son las palabras que la rodean en el contexto. (Ej: Dado "gato", adivinar "el", "persigue", "al", "ratón"). *Skip-gram suele funcionar mejor para datasets grandes y palabras raras.*

## 2. Toy Example (Cómo se generan los datos de entrenamiento)

Vamos a ver cómo **Skip-gram** prepara sus datos a partir de una frase simple usando una **ventana de contexto** (context window). 

Supongamos que nuestra ventana de contexto es de tamaño 2 (miramos 2 palabras hacia atrás y 2 hacia adelante).

**Frase:** "el rápido zorro marrón salta"

Paso a paso, deslizamos el "foco" sobre cada palabra:

| Palabra Central (Input) | Contexto (Ventana de tamaño 2) | Pares de Entrenamiento (Input $\rightarrow$ Target) |
| :--- | :--- | :--- |
| **el** | rápido, zorro | (el $\rightarrow$ rápido), (el $\rightarrow$ zorro) |
| **rápido** | el, zorro, marrón | (rápido $\rightarrow$ el), (rápido $\rightarrow$ zorro), (rápido $\rightarrow$ marrón) |
| **zorro** | el, rápido, marrón, salta | (zorro $\rightarrow$ el), (zorro $\rightarrow$ rápido), (zorro $\rightarrow$ marrón), (zorro $\rightarrow$ salta) |

**Conclusión del ejemplo:**
La red neuronal toma estos pares y se entrena para predecir el *Target* dado el *Input*. Aunque la red neuronal es "falsa" (no la usaremos para predecir palabras después), lo que nos importa son los **pesos de la capa oculta** que la red aprendió durante este proceso. Esos pesos son los vectores (embeddings) de nuestras palabras.

## 3. Implementación en Python

Para Word2Vec, la librería estándar en la industria es `gensim`. 

*(Si no la tienes, instálala con: `pip install gensim`)*

In [2]:
from gensim.models import Word2Vec

# 1. Nuestro corpus tokenizado (lista de listas de palabras)
# En la vida real, esto serían miles de artículos de Wikipedia o noticias
corpus = [
    ["el", "perro", "corre", "por", "el", "parque"],
    ["el", "gato", "duerme", "en", "el", "sofá"],
    ["un", "perro", "ladra", "a", "un", "gato"],
    ["el", "zorro", "salta", "sobre", "el", "perro"],
    ["un", "gato", "juega", "con", "un", "ratón"]
]

# 2. Entrenamos el modelo Word2Vec
# vector_size = dimensiones del vector (usamos 10 por ser un corpus diminuto, en prod es 100-300)
# window = tamaño de la ventana de contexto
# min_count = ignora palabras que aparezcan menos de X veces
# sg = 1 usa Skip-gram, sg = 0 usa CBOW
modelo = Word2Vec(sentences=corpus, vector_size=10, window=2, min_count=1, sg=1)

# 3. Ver el vector numérico (embedding) de una palabra
vector_perro = modelo.wv["perro"]
print("Vector de 'perro':\n", vector_perro, "\n")

# 4. Encontrar las palabras más similares matemáticamente
# (Calcula la similitud del coseno entre los vectores)
similares = modelo.wv.most_similar("perro", topn=2)
print("Palabras más similares a 'perro':")
for palabra, similitud in similares:
    print(f"- {palabra}: {similitud:.4f}")

Vector de 'perro':
 [-0.07511408 -0.00930313  0.09538089 -0.07318769 -0.02333963 -0.01937972
  0.08077365 -0.05930922  0.00044991 -0.04754165] 

Palabras más similares a 'perro':
- sobre: 0.2941
- por: 0.2896


Este párrafo describe exactamente la "magia" detrás de cómo aprenden los modelos como **Word2Vec** y cómo se construyen los **Embeddings** (vectores de palabras) en la inteligencia artificial moderna. 

En términos sencillos, explica cómo pasamos de un modelo ignorante a uno que entiende el significado matemático de las palabras. 

Vamos a desglosarlo en tres partes con una analogía (un *Toy Example* conceptual):

### 1. El Objetivo: El "Vector Space" (Espacio Vectorial)
> *"When we’re creating word vectors, the overarching concept is that we’d like to assign each word... to a particular, meaningful location within a multidimensional space..."*

**¿Qué quiere decir?**
Imagina una habitación vacía gigante. Esta habitación representa tu "espacio vectorial". El objetivo es tomar cada palabra de tu diccionario y asignarle una silla (una coordenada específica) dentro de esta habitación. Queremos que la ubicación de esa silla tenga sentido lógico.

### 2. El Inicio: Ubicaciones Aleatorias
> *"Initially, each word is assigned to a random location within the vector space."*

**¿Qué quiere decir?**
Antes de entrenar el modelo, la IA no sabe qué es un "perro" o qué es una "computadora". Así que toma todas las palabras y las tira al azar por toda la habitación. Por pura casualidad, la palabra "perro" podría caer justo al lado de "computadora", y muy lejos de "gato". En este punto, las coordenadas no tienen ningún significado semántico.

### 3. El Aprendizaje: Desplazamiento Gradual
> *"By considering the words that tend to be used around a given word... the locations of the words within the vector space can gradually be shifted into locations that represent the meaning..."*

**¿Qué quiere decir?**
Aquí es donde entra la regla del contexto (*"conocerás a una palabra por la compañía que frecuenta"*). El modelo empieza a leer miles de textos. 

Se da cuenta de que la palabra "perro" casi siempre está rodeada de palabras como "ladra", "mascota", "come" y "parque". Luego ve que la palabra "gato" *también* está rodeada de palabras similares. 

Al notar esto, el modelo hace un ajuste matemático: **empuja (desplaza)** ligeramente la palabra "perro" y la palabra "gato" para que se acerquen entre sí en esa gran habitación, y a su vez, las aleja de la palabra "computadora" (que tiene un contexto totalmente distinto, como "teclado" o "pantalla").

---

**En conclusión:** 
El texto explica que el entrenamiento de un modelo de lenguaje es, literalmente, el proceso de mover palabras de posiciones aleatorias a posiciones con significado dentro de un espacio matemático. Al final del entrenamiento, si mides la distancia entre "perro" y "gato" en ese espacio, estarán muy cerca; formarán un "vecindario" de mascotas.

Como se mencionó al principio de este capítulo, esta interpretación del significado de una palabra a partir de las palabras que la rodean fue propuesta por Ludwig Wittgenstein. Más tarde, en 1957, el lingüista británico J. R. Firth resumió esta idea de forma concisa con la frase: «Conocerás una palabra por la compañía que la rodea». Firth, J. (1957). Estudios de análisis lingüístico. Oxford: Blackwell.